# Tokenizer
Downloads the specified tokens from the Project Gutenberg dataset, tokenizes it using BPE and then saves the test,train and val datasets as .bin files

In [11]:
import json
import os
import time
import random
from pathlib import Path
from collections import Counter, defaultdict
from urllib.parse import urlparse
import http.client
import requests

import numpy as np
from tqdm.auto import tqdm

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

In [12]:
API_KEY = "828015eac9msh09a807a043bbecbp150f8cjsnab6ace59a400"

API_HOST = (
    "project-gutenberg-free-books-api1.p.rapidapi.com"
)

HEADERS = {
    "x-rapidapi-key": API_KEY,  
    "x-rapidapi-host": API_HOST,
    "Content-Type": "application/json",
}

MAX_API_REQUESTS = 850       # Leave 50 requests safety margin
MIN_REQUEST_INTERVAL = 0.25 # 4 req/sec max

API_STATE_FILE = Path(
    "data/gutenberg_dataset/api_state.json"
)

API_STATE_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)

In [13]:
def load_api_state():

    if API_STATE_FILE.exists():

        with open(
            API_STATE_FILE,
            "r",
            encoding="utf-8"
        ) as f:

            return json.load(f)

    return {
        "requests_used": 0,
        "last_request_time": 0.0,
    }


api_state = load_api_state()


def save_api_state():

    with open(
        API_STATE_FILE,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            api_state,
            f,
            indent=2
        )


def remaining_api_requests():

    return (
        MAX_API_REQUESTS
        - api_state["requests_used"]
    )

In [14]:
import re

GUTENBERG_START_RE = re.compile(
    r"\*\*\*\s*START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*",
    re.IGNORECASE | re.DOTALL,
)

GUTENBERG_END_RE = re.compile(
    r"\*\*\*\s*END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*",
    re.IGNORECASE | re.DOTALL,
)

def strip_gutenberg_boilerplate(text):
    start_match = GUTENBERG_START_RE.search(text)
    end_match = GUTENBERG_END_RE.search(text)

    if start_match:
        text = text[start_match.end():]

    if end_match:
        # end_match indices are relative to the ORIGINAL text,
        # so recompute the cut point relative to the trimmed text.
        end_pos = end_match.start()
        if start_match:
            end_pos -= start_match.end()
        text = text[:end_pos]

    return text.strip()

In [15]:
def api_get(endpoint, retries=5):
    if remaining_api_requests() <= 0:
        raise RuntimeError(
            "RapidAPI request budget exhausted. "
            f"Used {api_state['requests_used']} of {MAX_API_REQUESTS}."
        )

    url = f"https://{API_HOST}{endpoint}"

    for attempt in range(retries):
        elapsed = time.monotonic() - api_state["last_request_time"]
        if elapsed < MIN_REQUEST_INTERVAL:
            time.sleep(MIN_REQUEST_INTERVAL - elapsed)

        res = requests.get(url, headers=HEADERS, timeout=30)

        api_state["requests_used"] += 1
        api_state["last_request_time"] = time.monotonic()
        save_api_state()

        print(f"[API {api_state['requests_used']}/{MAX_API_REQUESTS}] "
            f"{res.status_code} {res.reason} | {endpoint}")

        if res.status_code == 200:
            return res.json()

        if res.status_code == 429:
            wait = min(2 ** attempt, 60)
            print(f"429 rate limit. Waiting {wait}s...")
            time.sleep(wait)
            continue

        print(res.text[:500])

        if res.status_code >= 500:
            wait = min(2 ** attempt, 30)
            time.sleep(wait)
            continue

        raise RuntimeError(f"API request failed: {res.status_code} {res.reason}")

    raise RuntimeError(f"Request failed after {retries} attempts.")

In [16]:
subjects = api_get(
    "/subjects?page=1"
)

subjects = subjects["results"]

print(
    f"Subjects retrieved: {len(subjects)}"
)

[API 86/850] 200 OK | /subjects?page=1
Subjects retrieved: 32


In [17]:
books_response = api_get(
    "/books?page=1"
)

book = books_response["results"][0]

print(
    json.dumps(
        book,
        indent=2
    )
)

[API 87/850] 200 OK | /books?page=1
{
  "id": 2701,
  "title": "Moby Dick; Or, The Whale",
  "alternative_title": null,
  "authors": [
    {
      "id": 505,
      "name": "Melville, Herman"
    }
  ],
  "subjects": [
    "Adventure stories",
    "Ahab, Captain (Fictitious character) -- Fiction",
    "Mentally ill -- Fiction",
    "Psychological fiction",
    "Sea stories",
    "Ship captains -- Fiction",
    "Whales -- Fiction",
    "Whaling -- Fiction",
    "Whaling ships -- Fiction"
  ],
  "bookshelves": [
    "Best Books Ever Listings",
    "Category: Adventure",
    "Category: American Literature",
    "Category: Classics of Literature",
    "Category: Novels"
  ],
  "formats": {
    "text/plain; charset=us-ascii": "https://www.gutenberg.org/files/2701/2701-0.txt",
    "text/plain": "https://www.gutenberg.org/files/2701/2701.txt",
    "text/html; charset=utf-8": "https://www.gutenberg.org/files/2701/2701-h.html",
    "text/html": "https://www.gutenberg.org/ebooks/2701.html.images"

In [18]:
subject_df = [
    {
        "id": s["id"],
        "name": s["name"],
        "books": s["book_count"],
        "downloads": s["download_count"],
    }
    for s in subjects
]

subject_df[:30]

for s in subjects:
    
    name = s["name"].lower()
    
    if any(
        word in name
        for word in [
            "fiction",
            "adventure",
            "mystery",
            "science",
            "history",
            "philosophy",
            "drama",
            "poetry",
        ]
    ):
        
        print(
            s["id"],
            "|",
            s["name"],
            "|",
            s["book_count"]
        )

1231095 | Science fiction | 3293
1231100 | Fiction | 1977
1231272 | Adventure stories | 1642
1231160 | Historical fiction | 1079
1231591 | Conduct of life -- Juvenile fiction | 1050
1231123 | Man-woman relationships -- Fiction | 1023
1231138 | Detective and mystery stories | 1013
1231313 | Poetry | 685
1231225 | England -- Fiction | 549
1232254 | Christian life -- Juvenile fiction | 521
1231101 | Young women -- Fiction | 495
1231637 | Fantasy fiction | 486
1231593 | Friendship -- Juvenile fiction | 486
1243888 | Human-alien encounters -- Fiction | 457
1231564 | Children -- Conduct of life -- Juvenile fiction | 447
1232525 | English fiction -- 19th century | 443
1232563 | Finnish fiction -- 20th century | 435
1231202 | Domestic fiction | 425
1231520 | French fiction -- 19th century | 406
1231201 | Psychological fiction | 388
1231575 | Animals -- Juvenile fiction | 357
1231594 | Siblings -- Juvenile fiction | 354


In [19]:
MAX_BOOK_PAGES = 65

def get_book_metadata_pages(max_pages=MAX_BOOK_PAGES):

    all_books = []
    endpoint = "/books?page=1"

    for _ in range(max_pages):

        data = api_get(endpoint)

        all_books.extend(
            data.get("results", [])
        )

        next_url = data.get("next")

        if not next_url:
            break

        parsed = urlparse(next_url)

        endpoint = parsed.path

        if parsed.query:
            endpoint += "?" + parsed.query

    return all_books

books = get_book_metadata_pages()

print(
    f"Books discovered: {len(books):,}"
)
print(
    f"RapidAPI requests used: "
    f"{api_state['requests_used']}"
)

[API 88/850] 200 OK | /books?page=1
[API 89/850] 200 OK | /books?page=2
[API 90/850] 200 OK | /books?page=3
[API 91/850] 200 OK | /books?page=4
[API 92/850] 200 OK | /books?page=5
[API 93/850] 200 OK | /books?page=6
[API 94/850] 200 OK | /books?page=7
[API 95/850] 200 OK | /books?page=8
[API 96/850] 200 OK | /books?page=9
[API 97/850] 200 OK | /books?page=10
[API 98/850] 200 OK | /books?page=11
[API 99/850] 200 OK | /books?page=12
[API 100/850] 200 OK | /books?page=13
[API 101/850] 200 OK | /books?page=14
[API 102/850] 200 OK | /books?page=15
[API 103/850] 200 OK | /books?page=16
[API 104/850] 200 OK | /books?page=17
[API 105/850] 200 OK | /books?page=18
[API 106/850] 200 OK | /books?page=19
[API 107/850] 200 OK | /books?page=20
[API 108/850] 200 OK | /books?page=21
[API 109/850] 200 OK | /books?page=22
[API 110/850] 200 OK | /books?page=23
[API 111/850] 200 OK | /books?page=24
[API 112/850] 200 OK | /books?page=25
[API 113/850] 200 OK | /books?page=26
[API 114/850] 200 OK | /books?pag

In [20]:
BAD_TERMS = [
    "dictionary",
    "catalog",
    "catalogue",
    "bibliography",
    "index",
    "encyclopedia",
    "encyclopaedia",
    "manual",
    "textbook",
    "grammar",
]

def is_good_book(book):
    
    title = (
        book.get("title")
        or ""
    ).lower()
    
    alternative_title = (
        book.get("alternative_title")
        or ""
    ).lower()
    
    subjects = " ".join(
        book.get(
            "subjects",
            []
        ) or []
    ).lower()
    
    bookshelves = " ".join(
        book.get(
            "bookshelves",
            []
        ) or []
    ).lower()
    
    combined = " ".join([
        title,
        alternative_title,
        subjects,
        bookshelves,
    ])
    
    for bad_term in BAD_TERMS:
        
        if bad_term in combined:
            return False
    
    return True

books = [
    book
    for book in books
    if is_good_book(book)
]

print(
    f"Remaining books: {len(books):,}"
)

Remaining books: 2,044


In [21]:
subject_counter = Counter()

for book in books:
    
    for subject in (
        book.get(
            "subjects",
            []
        ) or []
    ):
        
        subject_counter[
            subject
        ] += 1
        
for subject, count in (
    subject_counter.most_common(40)
):
    
    print(
        f"{count:6d} | {subject}"
    )

    95 | Detective and mystery stories
    79 | Science fiction
    69 | Love stories
    62 | Adventure stories
    60 | England -- Fiction
    59 | Short stories
    56 | Fiction
    55 | Historical fiction
    41 | Young women -- Fiction
    40 | Western stories
    38 | Domestic fiction
    33 | Man-woman relationships -- Fiction
    30 | Bildungsromans
    29 | Psychological fiction
    27 | Fantasy fiction
    25 | Private investigators -- England -- Fiction
    24 | Horror tales
    22 | London (England) -- Fiction
    22 | Conduct of life -- Juvenile fiction
    20 | Detective and mystery stories, English
    19 | Humorous stories
    19 | Inheritance and succession -- Fiction
    17 | Sea stories
    17 | Mystery fiction
    17 | New York (N.Y.) -- Fiction
    16 | Girls -- Juvenile fiction
    15 | Social classes -- Fiction
    15 | Orphans -- Fiction
    15 | Triangles (Interpersonal relations) -- Fiction
    14 | Sisters -- Fiction
    14 | Holmes, Sherlock (Fictitious char

In [22]:
GENRE_TERMS = {
    
    "fiction": [
        "fiction",
        "novels",
        "novel",
    ],
    
    "adventure": [
        "adventure",
        "adventure stories",
    ],
    
    "mystery": [
        "detective",
        "mystery",
        "crime",
    ],
    
    "scifi": [
        "science fiction",
        "science fiction stories",
    ],
    
    "horror": [
        "horror",
        "ghost stories",
        "supernatural",
    ],
    
    "romance": [
        "romance",
        "love stories",
    ],
    
    "history": [
        "history",
        "historical fiction",
        "historical",
    ],
    
    "philosophy": [
        "philosophy",
        "ethics",
        "metaphysics",
    ],
    
    "biography": [
        "biography",
        "autobiography",
        "memoirs",
    ],
    
    "travel": [
        "travel",
        "voyages",
        "exploration",
    ],
    
    "drama": [
        "drama",
        "plays",
        "tragedies",
        "comedies",
    ],
    
    "poetry": [
        "poetry",
        "poems",
        "poems",
    ],
}

In [23]:
def get_book_genres(book):
    
    text = " ".join(
        (
            book.get(
                "subjects",
                []
            ) or []
        )
        +
        (
            book.get(
                "bookshelves",
                []
            ) or []
        )
    ).lower()
    
    genres = []
    
    for genre, terms in GENRE_TERMS.items():
        
        if any(
            term.lower() in text
            for term in terms
        ):
            
            genres.append(
                genre
            )
    
    return genres


In [24]:
genre_pools = defaultdict(list)

for book in books:
    
    genres = get_book_genres(
        book
    )
    
    for genre in genres:
        
        genre_pools[
            genre
        ].append(book)

for genre, pool in genre_pools.items():
    
    print(
        f"{genre:15} "
        f"{len(pool):6,}"
    )

fiction          1,152
adventure          348
romance            283
mystery            263
drama               49
history            613
horror              53
scifi              100
poetry             108
biography          113
philosophy         102
travel             129


In [25]:
MAX_BOOKS_PER_AUTHOR = 15

author_counts = defaultdict(int)

candidate_books = []

random.seed(42)

# Shuffle first so equally ranked books don't always
# get selected in exactly the same order.
random.shuffle(books)

# Sort popular books toward the front.
books.sort(
    key=lambda b: (
        b.get(
            "download_count",
            0
        ) or 0
    ),
    reverse=True
)

for book in books:
    
    authors = book.get(
        "authors",
        []
    ) or []
    
    if authors:
        
        author = authors[0].get(
            "name",
            "Unknown"
        )
    
    else:
        
        author = "Unknown"
    
    if (
        author_counts[author]
        >= MAX_BOOKS_PER_AUTHOR
    ):
        continue
    
    candidate_books.append(
        book
    )
    
    author_counts[author] += 1

print(
    f"Candidate books: "
    f"{len(candidate_books):,}"
)

Candidate books: 1,952


In [26]:
CACHE_DIR = Path(
    "data/gutenberg_cache"
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [27]:
for f in CACHE_DIR.glob("*.txt"):
    text = f.read_text(encoding="utf-8", errors="ignore")
    f.write_text(strip_gutenberg_boilerplate(text), encoding="utf-8")

In [28]:
http_session = requests.Session()

def get_text_url(book):
    book_id = book["id"]

    return (
        f"https://www.gutenberg.org/cache/epub/"
        f"{book_id}/pg{book_id}.txt"
    )

def download_book(book):
    book_id = book["id"]
    cache_path = CACHE_DIR / f"{book_id}.txt"

    if cache_path.exists():
        return cache_path.read_text(encoding="utf-8", errors="ignore")

    url = get_text_url(book)

    try:
        response = http_session.get(url, timeout=60)
        response.raise_for_status()

        response.encoding = response.apparent_encoding  # fix #5 below
        text = response.text

        text = strip_gutenberg_boilerplate(text)  # <-- new

        cache_path.write_text(text, encoding="utf-8")
        return text

    except Exception as e:
        print(f"Failed {book_id}: {e}")
        return None

In [29]:
VOCAB_SIZE = 8_192
TOKENIZER_BOOK_COUNT = 200

random.seed(42)

tokenizer_books = random.sample(
    candidate_books,
    min(
        TOKENIZER_BOOK_COUNT,
        len(candidate_books)
    )
)

print(
    len(tokenizer_books)
)

200


In [30]:
tokenizer_texts = []

for book in tqdm(
    tokenizer_books,
    desc="Downloading tokenizer books"
):
    
    try:
        
        text = download_book(
            book
        )
        
        if text is not None:
            tokenizer_texts.append(
                text
            )
        
        time.sleep(0.1)
        
    except Exception as e:
        
        print(
            f"Failed {book['id']}: {e}"
        )

Failed 20968: 404 Client Error: Not Found for url: https://www.gutenberg.org/cache/epub/20968/pg20968.txt
Failed 19636: 404 Client Error: Not Found for url: https://www.gutenberg.org/cache/epub/19636/pg19636.txt
Failed 23712: 404 Client Error: Not Found for url: https://www.gutenberg.org/cache/epub/23712/pg23712.txt
Failed 26469: 404 Client Error: Not Found for url: https://www.gutenberg.org/cache/epub/26469/pg26469.txt
Failed 9695: 404 Client Error: Not Found for url: https://www.gutenberg.org/cache/epub/9695/pg9695.txt
Failed 20051: 404 Client Error: Not Found for url: https://www.gutenberg.org/cache/epub/20051/pg20051.txt


In [31]:
DATASET_DIR = Path(
    "data/gutenberg_dataset"
)

DATASET_DIR.mkdir(
    parents=True,
    exist_ok=True
)

tokenizer_training_file = (
    DATASET_DIR
    / "tokenizer_training.txt"
)

with open(
    tokenizer_training_file,
    "w",
    encoding="utf-8"
) as f:
    
    for text in tokenizer_texts:
        
        f.write(text)
        
        f.write(
            "\n<|endoftext|>\n"
        )

In [32]:
tokenizer = Tokenizer(
    BPE(
        unk_token="<unk>"
    )
)

tokenizer.pre_tokenizer = ByteLevel(
    add_prefix_space=False
)

tokenizer.decoder = ByteLevelDecoder()

trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    
    min_frequency=2,
    
    special_tokens=[
        "<unk>",
        "<|endoftext|>",
    ],
    
    initial_alphabet=ByteLevel.alphabet(),
)

tokenizer.train(
    files=[
        str(
            tokenizer_training_file
        )
    ],
    trainer=trainer,
)

print(
    "Vocabulary size:",
    tokenizer.get_vocab_size()
)

TOKENIZER_PATH = (
    DATASET_DIR
    / "tokenizer.json"
)

tokenizer.save(
    str(TOKENIZER_PATH)
)

Vocabulary size: 8192


In [33]:
book_stats = []

for book in tqdm(
    candidate_books,
    desc="Counting BPE tokens"
):
    
    try:
        
        text = download_book(
            book
        )
        
        if text is None:
            continue
        
        # Avoid obviously tiny texts.
        if len(text) < 20_000:
            continue
        
        encoding = tokenizer.encode(
            text,
            add_special_tokens=False
        )
        
        token_count = (
            len(encoding.ids)
            + 1
        )
        
        book_stats.append({
            "book": book,
            "tokens": token_count
        })
        
        time.sleep(0.05)
        
    except Exception as e:
        
        print(
            f"Failed {book['id']}: {e}"
        )

Counting BPE tokens:   0%|          | 0/1952 [00:00<?, ?it/s]

Failed 26471: 404 Client Error: Not Found for url: https://www.gutenberg.org/cache/epub/26471/pg26471.txt
Failed 26296: 404 Client Error: Not Found for url: https://www.gutenberg.org/cache/epub/26296/pg26296.txt
Failed 26225: 404 Client Error: Not Found for url: https://www.gutenberg.org/cache/epub/26225/pg26225.txt
Failed 28942: 404 Client Error: Not Found for url: https://www.gutenberg.org/cache/epub/28942/pg28942.txt
Failed 22789: 404 Client Error: Not Found for url: https://www.gutenberg.org/cache/epub/22789/pg22789.txt
Failed 20968: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
Failed 22788: 404 Client Error: Not Found for url: https://www.gutenberg.org/cache/epub/22788/pg22788.txt
Failed 20000: 404 Client Error: Not Found for url: https://www.gutenberg.org/cache/epub/20000/pg20000.txt
Failed 26301: 404 Client Error: Not Found for url: https://www.gutenberg.org/cache/epub/26301/pg26301.txt
Failed 21076: 404 Client Error: Not Found for

In [34]:
TRAIN_TARGET = 200_000_000
VAL_TARGET = 10_000_000
TEST_TARGET = 10_000_000

TOTAL_TARGET = (
    TRAIN_TARGET
    + VAL_TARGET
    + TEST_TARGET
)

print(
    f"Required total: "
    f"{TOTAL_TARGET:,}"
)

Required total: 220,000,000


In [35]:
available_tokens = sum(x["tokens"] for x in book_stats)
print(f"Available: {available_tokens:,} BPE tokens")

assert available_tokens >= TOTAL_TARGET, (
    f"Only {available_tokens:,} tokens available, "
    f"but {TOTAL_TARGET:,} required. "
    f"Increase MAX_BOOK_PAGES or relax filters."
)

Available: 347,408,456 BPE tokens


In [36]:
random.seed(42)

random.shuffle(
    book_stats
)

In [37]:
val_books = []
val_tokens = 0

for item in book_stats:
    
    if val_tokens >= VAL_TARGET:
        break
    
    val_books.append(
        item
    )
    
    val_tokens += item["tokens"]

In [38]:
used_ids = {
    item["book"]["id"]
    for item in val_books
}

test_books = []
test_tokens = 0

for item in book_stats:
    
    if item["book"]["id"] in used_ids:
        continue
    
    if test_tokens >= TEST_TARGET:
        break
    
    test_books.append(
        item
    )
    
    test_tokens += item["tokens"]

In [39]:
used_ids.update(
    item["book"]["id"]
    for item in test_books
)

train_books = []
train_tokens = 0

for item in book_stats:
    
    if item["book"]["id"] in used_ids:
        continue
    
    if train_tokens >= TRAIN_TARGET:
        break
    
    train_books.append(
        item
    )
    
    train_tokens += item["tokens"]

In [40]:
print(
    f"TRAIN: "
    f"{train_tokens:,} tokens "
    f"from {len(train_books)} books"
)

print(
    f"VAL:   "
    f"{val_tokens:,} tokens "
    f"from {len(val_books)} books"
)

print(
    f"TEST:  "
    f"{test_tokens:,} tokens "
    f"from {len(test_books)} books"
)

TRAIN: 200,097,153 tokens from 1057 books
VAL:   10,200,685 tokens from 55 books
TEST:  10,140,394 tokens from 55 books


In [41]:
for item in train_books[:30]:
    
    book = item["book"]
    
    authors = ", ".join(
        a["name"]
        for a in (
            book.get(
                "authors",
                []
            ) or []
        )
    )
    
    print(
        f"{book['id']:>6} | "
        f"{item['tokens']:>10,} | "
        f"{book['title'][:60]} | "
        f"{authors}"
    )

 22066 |    468,318 | The Long Roll | Johnston, Mary
 16199 |     40,461 | Memoirs of the Author of a Vindication of the Rights of Woma | Godwin, William
 21597 |     49,107 | Jersey Street and Jersey Lane: Urban and Suburban Sketches | Bunner, H. C. (Henry Cuyler)
 34318 |    178,953 | Life Blood | Hoover, Thomas
 21655 |    177,603 | The works of Guy de Maupassant, Vol. 5: Une Vie and Other St | Maupassant, Guy de
 27488 |    226,530 | Lectures on the French Revolution | Acton, John Emerich Edward Dalberg Acton, Baron
 25656 |     52,014 | The Mystery of a Turkish Bath | Rita
 50603 |     50,266 | Minute Mysteries [Detectograms] | Ripley, H. A. (Harold Austin)
   831 |    293,663 | Four Arthurian Romances | Chrétien, de Troyes, active 12th century
 21219 |    160,955 | A Voice in the Wilderness | Hill, Grace Livingston
 23747 |    170,858 | Destruction and Reconstruction:: Personal Experiences of the | Taylor, Richard
 22186 |    106,702 | For the Liberty of Texas | Stratemeyer, Edwa

In [42]:
def save_split(
    items,
    filename
):
    
    data = []
    
    for item in items:
        
        book = item["book"]
        
        data.append({
            "id": book["id"],
            "title": book["title"],
            "authors": book.get(
                "authors",
                []
            ),
            "subjects": book.get(
                "subjects",
                []
            ),
            "bookshelves": book.get(
                "bookshelves",
                []
            ),
            "tokens": item["tokens"],
        })
    
    path = (
        DATASET_DIR
        / filename
    )
    
    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:
        
        json.dump(
            data,
            f,
            indent=2
        )
    
    print(path)

save_split(
    train_books,
    "train_books.json"
)

save_split(
    val_books,
    "val_books.json"
)

save_split(
    test_books,
    "test_books.json"
)

data\gutenberg_dataset\train_books.json
data\gutenberg_dataset\val_books.json
data\gutenberg_dataset\test_books.json


In [43]:
for item in train_books[:10]:

    book_id = item["book"]["id"]

    cache_path = (
        CACHE_DIR
        / f"{book_id}.txt"
    )

    text = cache_path.read_text(
        encoding="utf-8",
        errors="ignore"
    )

    actual = (
        len(
            tokenizer.encode(
                text,
                add_special_tokens=False
            ).ids
        )
        + 1
    )

    recorded = item["tokens"]

    if actual != recorded:

        print(
            f"Mismatch: {book_id} | "
            f"recorded={recorded:,} | "
            f"actual={actual:,}"
        )

In [44]:
def create_token_file(
    items,
    output_filename
):

    print(
        f"\nPreparing {output_filename}..."
    )

    # ========================================================
    # 1. Encode every book and keep the encoded IDs
    # ========================================================

    encoded_books = []

    total_tokens = 0

    eos_id = tokenizer.token_to_id(
        "<|endoftext|>"
    )

    if eos_id is None:
        raise RuntimeError(
            "<|endoftext|> token not found."
        )

    for item in tqdm(
        items,
        desc="Encoding selected books"
    ):

        book_id = item["book"]["id"]

        cache_path = (
            CACHE_DIR
            / f"{book_id}.txt"
        )

        if not cache_path.exists():

            raise FileNotFoundError(
                f"Cached text not found:\n"
                f"{cache_path}"
            )

        text = cache_path.read_text(
            encoding="utf-8",
            errors="ignore"
        )

        encoding = tokenizer.encode(
            text,
            add_special_tokens=False
        )

        ids = np.asarray(
            encoding.ids,
            dtype=np.uint16
        )

        # +1 for EOS
        encoded_length = (
            len(ids) + 1
        )

        encoded_books.append({
            "book_id": book_id,
            "ids": ids,
            "length": encoded_length,
        })

        total_tokens += encoded_length

    print(
        f"Exact token count: "
        f"{total_tokens:,}"
    )

    # ========================================================
    # 2. Allocate exact memmap
    # ========================================================

    output_path = (
        DATASET_DIR
        / output_filename
    )

    arr = np.memmap(
        output_path,
        dtype=np.uint16,
        mode="w+",
        shape=(total_tokens,)
    )

    # ========================================================
    # 3. Write
    # ========================================================

    position = 0

    for item in tqdm(
        encoded_books,
        desc=f"Writing {output_filename}"
    ):

        ids = item["ids"]

        end = (
            position
            + len(ids)
        )

        arr[
            position:end
        ] = ids

        position = end

        # EOS
        arr[position] = eos_id

        position += 1

    # ========================================================
    # 4. Verify
    # ========================================================

    if position != total_tokens:

        raise RuntimeError(
            f"Write mismatch!\n"
            f"Expected: {total_tokens:,}\n"
            f"Wrote:    {position:,}"
        )

    arr.flush()

    del arr

    print(
        f"Saved: {output_path}"
    )

    print(
        f"Size: "
        f"{output_path.stat().st_size / 1024**3:.2f} GB"
    )

    return output_path

train_path = create_token_file(
    train_books,
    "train.bin"
)

val_path = create_token_file(
    val_books,
    "val.bin"
)

test_path = create_token_file(
    test_books,
    "test.bin"
)


Preparing train.bin...


Encoding selected books:   0%|          | 0/1057 [00:00<?, ?it/s]

Exact token count: 200,101,711


Writing train.bin:   0%|          | 0/1057 [00:00<?, ?it/s]

Saved: data\gutenberg_dataset\train.bin
Size: 0.37 GB

Preparing val.bin...


Encoding selected books:   0%|          | 0/55 [00:00<?, ?it/s]

... (3 outputs left out)

Encoding selected books:   0%|          | 0/55 [00:00<?, ?it/s]

Exact token count: 10,140,394


Writing test.bin:   0%|          | 0/55 [00:00<?, ?it/s]

Saved: data\gutenberg_dataset\test.bin
Size: 0.02 GB


In [45]:
print("=" * 60)
print("GUTENBERG DATASET")
print("=" * 60)

print(
    f"Vocabulary: "
    f"{tokenizer.get_vocab_size():,}"
)

print()

print(
    f"Train tokens: "
    f"{train_tokens:,}"
)

print(
    f"Validation tokens: "
    f"{val_tokens:,}"
)

print(
    f"Test tokens: "
    f"{test_tokens:,}"
)

print()

print(
    f"Train books: "
    f"{len(train_books):,}"
)

print(
    f"Validation books: "
    f"{len(val_books):,}"
)

print(
    f"Test books: "
    f"{len(test_books):,}"
)

GUTENBERG DATASET
Vocabulary: 8,192

Train tokens: 200,097,153
Validation tokens: 10,200,685
Test tokens: 10,140,394

Train books: 1,057
Validation books: 55
Test books: 55


In [46]:
import numpy as np
from pathlib import Path

def inspect_bin(path):
    path = Path(path)

    size_bytes = path.stat().st_size

    # uint16 = 2 bytes
    token_count = size_bytes // np.dtype(np.uint16).itemsize

    print(f"{path}")
    print(f"Bytes:  {size_bytes:,}")
    print(f"Tokens: {token_count:,}")

    return token_count


train_bin_tokens = inspect_bin(
    DATASET_DIR / "train.bin"
)

val_bin_tokens = inspect_bin(
    DATASET_DIR / "val.bin"
)

test_bin_tokens = inspect_bin(
    DATASET_DIR / "test.bin"
)

data\gutenberg_dataset\train.bin
Bytes:  400,203,422
Tokens: 200,101,711
data\gutenberg_dataset\val.bin
Bytes:  20,401,370
Tokens: 10,200,685
data\gutenberg_dataset\test.bin
Bytes:  20,280,788
Tokens: 10,140,394
